# 🏥 Healthcare No-Show Predictor
## Notebook 2: Data Preprocessing

**Goal:** Clean and prepare data for ML models.

**Steps:**
1. Fix data quality issues (negative ages, waiting days)
2. Encode categorical variables
3. Engineer new features
4. Scale features
5. Split into train and test sets
6. Save clean data for modeling

In [ ]:
# ── Imports ──────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

%matplotlib inline
pd.set_option('display.max_columns', None)

print("✅ Libraries loaded")

In [ ]:
# ── Load Raw Data ─────────────────────────────────────
df = pd.read_csv('../data/KaggleV2-May-2016.csv')

# Rename columns (same as EDA)
df = df.rename(columns={
    'PatientId'     : 'patient_id',
    'AppointmentID' : 'appointment_id',
    'Gender'        : 'gender',
    'ScheduledDay'  : 'scheduled_day',
    'AppointmentDay': 'appointment_day',
    'Age'           : 'age',
    'Neighbourhood' : 'neighbourhood',
    'Scholarship'   : 'scholarship',
    'Hipertension'  : 'hypertension',
    'Diabetes'      : 'diabetes',
    'Alcoholism'    : 'alcoholism',
    'Handcap'       : 'handicap',
    'SMS_received'  : 'sms_received',
    'No-show'       : 'no_show'
})

print(f"Raw data shape: {df.shape}")
print("✅ Data loaded and columns renamed")

In [ ]:
# ── Feature Engineering: Waiting Days ────────────────
df['scheduled_day']   = pd.to_datetime(df['scheduled_day'])
df['appointment_day'] = pd.to_datetime(df['appointment_day'])

df['waiting_days'] = (df['appointment_day'] - 
                      df['scheduled_day']).dt.days

print("=== WAITING DAYS BEFORE CLEANING ===")
print(f"Negative waiting days : {len(df[df['waiting_days'] < 0])}")
print(f"Min waiting days      : {df['waiting_days'].min()}")
print(f"Max waiting days      : {df['waiting_days'].max()}")

In [ ]:
# ── Fix Data Quality Issues ───────────────────────────
print("=== BEFORE CLEANING ===")
print(f"Total rows           : {len(df):,}")
print(f"Negative ages        : {len(df[df['age'] < 0])}")
print(f"Negative waiting days: {len(df[df['waiting_days'] < 0])}")

# Fix 1 — Remove negative ages (impossible in real life)
df = df[df['age'] >= 0]

# Fix 2 — Remove negative waiting days (data errors)
df = df[df['waiting_days'] >= 0]

# Fix 3 — Cap waiting days at 180 (6 months — extreme outliers)
df = df[df['waiting_days'] <= 180]

print("\n=== AFTER CLEANING ===")
print(f"Total rows remaining : {len(df):,}")
print(f"Rows removed         : {110527 - len(df):,}")
print(f"Negative ages        : {len(df[df['age'] < 0])}")
print(f"Negative waiting days: {len(df[df['waiting_days'] < 0])}")
print("✅ Data quality issues fixed")

In [ ]:
# ── Encode Target Variable ────────────────────────────
# Convert No-show: Yes → 1, No → 0
df['no_show_binary'] = (df['no_show'] == 'Yes').astype(int)

print("=== TARGET ENCODING ===")
print(df['no_show'].value_counts())
print("\nAfter encoding:")
print(df['no_show_binary'].value_counts())
print(f"\n0 = showed up")
print(f"1 = no show")
print("✅ Target encoded")

In [ ]:
# ── Encode Categorical Features ───────────────────────
# Gender: F → 0, M → 1
df['gender_binary'] = (df['gender'] == 'M').astype(int)

print("=== GENDER ENCODING ===")
print(df['gender'].value_counts())
print("\nAfter encoding:")
print(df['gender_binary'].value_counts())
print("\n0 = Female")
print("1 = Male")
print("✅ Gender encoded")

In [ ]:
# ── Additional Feature Engineering ───────────────────
# Age groups — younger patients no-show more
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 18, 35, 60, 120],
    labels=[0, 1, 2, 3]   # child, young adult, adult, senior
).astype(int)

# Same day appointment (waiting_days = 0)
df['same_day'] = (df['waiting_days'] == 0).astype(int)

# Appointment day of week
df['appt_weekday'] = df['appointment_day'].dt.dayofweek
# 0=Monday, 4=Friday, 5=Saturday

print("=== NEW FEATURES ===")
print(f"age_group distribution:\n{df['age_group'].value_counts().sort_index()}")
print(f"\nsame_day appointments: {df['same_day'].sum():,}")
print(f"weekday distribution:\n{df['appt_weekday'].value_counts().sort_index()}")
print("✅ New features engineered")

In [ ]:
# ── Select Features for ML ────────────────────────────
feature_cols = [
    'age',            # patient age
    'gender_binary',  # gender encoded
    'scholarship',    # welfare enrollment
    'hypertension',   # has hypertension
    'diabetes',       # has diabetes
    'alcoholism',     # has alcoholism
    'handicap',       # handicap level
    'sms_received',   # received reminder
    'waiting_days',   # days between schedule and appointment
    'age_group',      # age category
    'same_day',       # same day appointment flag
    'appt_weekday'    # day of week
]

target_col = 'no_show_binary'

X = df[feature_cols]
y = df[target_col]

print("=== FEATURES SELECTED ===")
print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape : {y.shape}")
print(f"\nFeatures used:")
for i, col in enumerate(feature_cols):
    print(f"  {i+1:2}. {col}")
print(f"\nTarget: {target_col}")
print(f"  0 = showed up  ({(y==0).sum():,})")
print(f"  1 = no show    ({(y==1).sum():,})")

In [ ]:
# ── Train Test Split ──────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 80% train, 20% test
    random_state=42,     # reproducible results
    stratify=y           # keep class balance in both splits
)

print("=== TRAIN TEST SPLIT ===")
print(f"Total samples  : {len(X):,}")
print(f"Training set   : {len(X_train):,} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test set       : {len(X_test):,}  ({len(X_test)/len(X)*100:.0f}%)")

print(f"\nClass balance in training set:")
print(f"  Showed up : {(y_train==0).sum():,} ({(y_train==0).mean()*100:.1f}%)")
print(f"  No show   : {(y_train==1).sum():,} ({(y_train==1).mean()*100:.1f}%)")

print(f"\nClass balance in test set:")
print(f"  Showed up : {(y_test==0).sum():,} ({(y_test==0).mean()*100:.1f}%)")
print(f"  No show   : {(y_test==1).sum():,} ({(y_test==1).mean()*100:.1f}%)")
print("\n✅ Stratified split complete")

In [ ]:
# ── Feature Scaling ───────────────────────────────────
# Z-score standardization (Lesson 2 in statistics!)
# mean=0, std=1 for each feature

scaler = StandardScaler()

# Fit on TRAINING data only — very important!
# Never fit on test data → data leakage!
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("=== FEATURE SCALING ===")
print("Before scaling (training set):")
print(pd.DataFrame(X_train, 
      columns=feature_cols).describe().round(2))

print("\nAfter scaling (training set):")
print(pd.DataFrame(X_train_scaled,
      columns=feature_cols).describe().round(2))
print("\n✅ Features scaled — mean≈0, std≈1")

In [ ]:
# ── Verify Scaling ────────────────────────────────────
print("=== SCALING VERIFICATION ===")
print(f"{'Feature':<15} {'Mean Before':>12} {'Mean After':>12} {'Std After':>12}")
print("─" * 55)

for i, col in enumerate(feature_cols):
    mean_before = X_train[col].mean()
    mean_after  = X_train_scaled[:, i].mean()
    std_after   = X_train_scaled[:, i].std()
    print(f"{col:<15} {mean_before:>12.2f} "
          f"{mean_after:>12.4f} {std_after:>12.4f}")

print("\n✅ All features: mean≈0, std≈1")

In [ ]:
# ── Save Processed Data ───────────────────────────────
import os

# Save to data folder
np.save('../data/X_train.npy', X_train_scaled)
np.save('../data/X_test.npy',  X_test_scaled)
np.save('../data/y_train.npy', y_train.values)
np.save('../data/y_test.npy',  y_test.values)

# Save feature names for reference
pd.Series(feature_cols).to_csv(
    '../data/feature_names.csv', index=False)

print("=== FILES SAVED ===")
print("✅ X_train.npy")
print("✅ X_test.npy")
print("✅ y_train.npy")
print("✅ y_test.npy")
print("✅ feature_names.csv")
print("\nReady for 03_modeling.ipynb! 🚀")

In [ ]:
## 📋 Preprocessing Summary

### Data Cleaning
- ✅ Removed negative ages
- ✅ Removed negative waiting days  
- ✅ Capped waiting days at 180

### Encoding
- ✅ Target: no_show → 0/1
- ✅ Gender: F/M → 0/1

### Feature Engineering
- ✅ waiting_days (from date subtraction)
- ✅ age_group (binned age)
- ✅ same_day (waiting_days == 0)
- ✅ appt_weekday (day of week)

### Splitting & Scaling
- ✅ 80/20 train/test split
- ✅ Stratified to preserve class balance
- ✅ Z-score standardization (fit on train only)

### Ready for Modeling
- Training set: ~88,000 samples
- Test set:     ~22,000 samples  
- Features:     12 dimensions
- Target:       Binary (0/1)